# Multi-Scholar Fatwas — Embedding Generation on Kaggle (2x T4 GPU)

## 📋 INSTRUCTIONS FOR RUNNING ON KAGGLE (PLEASE READ FIRST)

Follow these steps to generate the embeddings for the full 40,000+ multi-scholar corpus on Kaggle:

### Step 1: Create a Kaggle Notebook
1. Open [kaggle.com](https://www.kaggle.com) and click **+ Create** -> **New Notebook**.
2. In the notebook editor, go to the right sidebar under **Settings**:
   - **Accelerator**: Select **GPU T4 x2** (recommended for 2x faster encoding) or **GPU P100**.
   - **Internet**: Ensure **Internet On** is checked (required to fetch `Qwen/Qwen3-Embedding-0.6B`).

### Step 2: Upload the Cleaned Dataset (`multi_scholar_clean.csv`)
1. In the right panel, click **+ Add Data** -> **Upload**.
2. Upload `data/cleaned/multi_scholar_clean.csv` (from this repository) and name the dataset `multi-scholar-fatwas`.
3. Once uploaded, the file will be located at `/kaggle/input/multi-scholar-fatwas/multi_scholar_clean.csv`.
*(Alternatively, cell 3 below automatically supports direct upload or fallback download).* 

### Step 3: Run All Cells
1. Click **Run All** (or run cell by cell).
2. Encoding ~40,000 fatwas with `Qwen3-Embedding-0.6B` on 2x T4 takes approximately 10–15 minutes.

### Step 4: Download the Output Artifacts
1. When the run finishes, check the right panel under **Output** (`/kaggle/working`).
2. You will see three files:
   - `multi_scholar_clean.csv`
   - `qwen3_06b_multi_scholar_embeddings.pt`
   - `multi_scholar_metadata.json`
3. Click the `...` menu next to `/kaggle/working` and select **Download all**, or download each file individually.
4. Move these three files into your local project's `data/` folder.
---


In [ ]:
# 1. Setup & Environment
!pip install -q sentence-transformers torch pandas

import os, time, json, gc
from datetime import datetime, timezone
from pathlib import Path
import torch
import pandas as pd
from sentence_transformers import SentenceTransformer

print(f'PyTorch version: {torch.__version__}')
n_gpus = torch.cuda.device_count()
print(f'CUDA available: {torch.cuda.is_available()} | GPUs count: {n_gpus}')
for i in range(n_gpus):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')
assert n_gpus > 0, 'No GPU detected! In Kaggle Settings -> Accelerator, select GPU T4 x2.'

In [ ]:
# 2. Configuration & Paths
WORK_DIR = Path('/kaggle/working')
INPUT_DIR = Path('/kaggle/input')

# Model configuration
EMBEDDER_MODEL_NAME = 'Qwen/Qwen3-Embedding-0.6B'
MAX_SEQ_LENGTH = 1024  # Cap sequence length for retrieval & reranking consistency
BATCH_SIZE = 32

# Output file paths
CLEAN_CSV_OUT = WORK_DIR / 'multi_scholar_clean.csv'
EMBEDDINGS_PT_OUT = WORK_DIR / 'qwen3_06b_multi_scholar_embeddings.pt'
METADATA_JSON_OUT = WORK_DIR / 'multi_scholar_metadata.json'

print('Configuration initialized.')

In [ ]:
# 3. Locate and Load multi_scholar_clean.csv
# Search for multi_scholar_clean.csv in /kaggle/input or local fallback
candidate_paths = list(INPUT_DIR.glob('**/multi_scholar_clean.csv')) + [
    Path('data/cleaned/multi_scholar_clean.csv'),
    Path('multi_scholar_clean.csv'),
]

csv_path = None
for p in candidate_paths:
    if p.exists():
        csv_path = p
        break

if csv_path is None:
    raise FileNotFoundError(
        'Could not find multi_scholar_clean.csv! '
        'Please add your uploaded dataset via the "+ Add Data" button on the right.'
    )

print(f'Loading dataset from: {csv_path}')
df = pd.read_csv(csv_path)
print(f'Loaded {len(df):,} total fatwas.')
print('\nDistribution by Scholar:')
print(df['source'].value_counts())

In [ ]:
# 4. Prepare Embed Texts and Check Truncation Statistics
# Standard retrieval unit: title + ' — ' + answer
if 'embed_text' not in df.columns or df['embed_text'].isnull().any():
    df['embed_text'] = df['title'].fillna('') + ' — ' + df['answer'].fillna('')

print('Loading tokenizer to check token length distributions...')
tokenizer = SentenceTransformer(EMBEDDER_MODEL_NAME, device='cpu').tokenizer

texts = df['embed_text'].astype(str).tolist()
token_counts = [len(ids) for ids in tokenizer(texts, add_special_tokens=False)['input_ids']]
df['embed_token_count'] = token_counts
df['truncated'] = df['embed_token_count'] > MAX_SEQ_LENGTH

n_trunc = int(df['truncated'].sum())
pct_trunc = n_trunc / len(df) * 100
print(f'Token length statistics (budget = {MAX_SEQ_LENGTH}):')
print(f'  Median: {pd.Series(token_counts).median():.0f} tokens')
print(f'  Mean:   {pd.Series(token_counts).mean():.1f} tokens')
print(f'  Max:    {max(token_counts):,} tokens')
print(f'  Truncated (> {MAX_SEQ_LENGTH}): {n_trunc:,} / {len(df):,} ({pct_trunc:.2f}%)')

del tokenizer
gc.collect()

In [ ]:
# 5. Multi-GPU Embedding Generation
print(f'Initializing embedder {EMBEDDER_MODEL_NAME} on GPU...')
embedder = SentenceTransformer(EMBEDDER_MODEL_NAME, device='cuda')
embedder.max_seq_length = MAX_SEQ_LENGTH

print(f'Starting embedding generation for {len(texts):,} texts...')
t0 = time.time()

if torch.cuda.device_count() >= 2:
    print('Using multi-process pool across all available GPUs...')
    pool = embedder.start_multi_process_pool()
    embeddings_np = embedder.encode_multi_process(
        texts,
        pool,
        batch_size=BATCH_SIZE,
        normalize_embeddings=True,
    )
    embedder.stop_multi_process_pool(pool)
else:
    print('Encoding on single GPU...')
    embeddings_np = embedder.encode(
        texts,
        batch_size=BATCH_SIZE,
        normalize_embeddings=True,
        show_progress_bar=True,
    )

elapsed = time.time() - t0
embeddings = torch.as_tensor(embeddings_np, dtype=torch.float32)
print(f'Encoding complete in {elapsed / 60:.2f} minutes!')
print(f'Embeddings shape: {tuple(embeddings.shape)}')
assert embeddings.shape[0] == len(df), 'Row count and embedding count mismatch!'

In [ ]:
# 6. Save Artifacts for Download
print(f'Saving cleaned CSV -> {CLEAN_CSV_OUT}')
df.to_csv(CLEAN_CSV_OUT, index=False)

print(f'Saving embeddings tensor -> {EMBEDDINGS_PT_OUT}')
torch.save(embeddings, EMBEDDINGS_PT_OUT)

metadata = {
    'model': EMBEDDER_MODEL_NAME,
    'embed_dim': int(embeddings.shape[1]),
    'max_seq_length': MAX_SEQ_LENGTH,
    'n_rows': int(embeddings.shape[0]),
    'n_truncated': n_trunc,
    'scholars': df['source'].value_counts().to_dict(),
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
}

print(f'Saving metadata JSON -> {METADATA_JSON_OUT}')
with open(METADATA_JSON_OUT, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print('\n' + '='*60)
print('SUCCESS! ALL 3 ARTIFACTS ARE READY IN /kaggle/working:')
print(f'  1. {CLEAN_CSV_OUT} ({CLEAN_CSV_OUT.stat().st_size / (1024*1024):.1f} MB)')
print(f'  2. {EMBEDDINGS_PT_OUT} ({EMBEDDINGS_PT_OUT.stat().st_size / (1024*1024):.1f} MB)')
print(f'  3. {METADATA_JSON_OUT}')
print('='*60)
print('Please download these 3 files from the Output tab and place them into your data/ folder.')
print(json.dumps(metadata, ensure_ascii=False, indent=2))